In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

### Change in the phase transition with respect to the decay parameter λ

In [4]:
import os
from pathlib import Path
from mintrans_clean import *



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

l = [0.07, 0.08, 0.09, 1]

checkpoint_dir = './skeletons'
if not os.path.exists(checkpoint_dir): 
    os.makedirs(checkpoint_dir)
# general_paramater in path refers to the configuration in minitrans_clean.py with different weight decay



# ---------------------------------------------------------------------------
# General configuration
# ---------------------------------------------------------------------------

cfg = TrainConfig

# ---------------------------------------------------------------------------
# Data loader configuration
# ---------------------------------------------------------------------------

torch.manual_seed(cfg.torch_seed)

train_ds = FibonacciTrainDataset(
    mod=cfg.p, seq_len=cfg.seq_len, train_frac=cfg.train_frac,
    seed=cfg.data_seed,
)
val_ds = FibonacciValDataset(train_ds, num_samples=500)

n_unseen = sum(
    1 for a in range(cfg.p) for b in range(cfg.p)
    if (a, b) not in train_ds.seen_pairs
)

cfg.batch_size = len(train_ds)   
cfg.epochs = 12000

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

# ---------------------------------------------------------------------------
# Model configuration 
# ---------------------------------------------------------------------------

model = MinimalTransformer(
    vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
    num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
    hidden_mlp=cfg.hidden_mlp,
).to(device)


for i in range(len(l)):
    weight_decay = l[i]

    cfg.weight_decay = weight_decay
    cfg.checkpoint_path = os.path.join(checkpoint_dir, f'general_paramater_wd_{weight_decay}.pth')

    try:
        history = train_model(model, train_loader, val_loader, cfg)
    except KeyboardInterrupt:
        history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc", "spectral")}

    val_loss, val_acc     = evaluate(model, val_loader)
    train_loss, train_acc = evaluate(model, train_loader)

    print(f"\nFinal — train acc: {train_acc:.3f}  val acc: {val_acc:.3f}")

    if cfg.checkpoint_path:
        save_checkpoint(model, history, cfg.checkpoint_path)



Epoch   100/12000  train loss 1.8410 acc 0.585  val loss 8.2107 acc 0.038
Epoch   200/12000  train loss 0.0054 acc 1.000  val loss 18.6211 acc 0.102
Epoch   300/12000  train loss 0.0013 acc 1.000  val loss 20.8109 acc 0.106
Epoch   400/12000  train loss 0.0004 acc 1.000  val loss 22.7247 acc 0.108
Epoch   500/12000  train loss 0.0001 acc 1.000  val loss 24.5421 acc 0.110
Epoch   600/12000  train loss 0.0000 acc 1.000  val loss 26.2952 acc 0.112
Epoch   700/12000  train loss 0.0000 acc 1.000  val loss 28.0081 acc 0.112
Epoch   800/12000  train loss 0.0000 acc 1.000  val loss 29.6827 acc 0.116
Epoch   900/12000  train loss 0.0000 acc 1.000  val loss 31.2978 acc 0.118
Epoch  1000/12000  train loss 0.0000 acc 1.000  val loss 32.8310 acc 0.118
Epoch  1100/12000  train loss 0.0000 acc 1.000  val loss 34.2210 acc 0.120
Epoch  1200/12000  train loss 0.0000 acc 1.000  val loss 35.3796 acc 0.120
Epoch  1300/12000  train loss 0.0000 acc 1.000  val loss 36.2173 acc 0.122
Epoch  1400/12000  train l